In [1]:
import os
import datetime
import subprocess
from lucifex.io import create_dir_path, dir_path_exists
from lucifex.sim.parallel import combine_options
from lucifex.utils.py_utils import FrozenDict
from crocodil.dns.system_a import SYSTEM_A_REFERENCE

Ra_ref = SYSTEM_A_REFERENCE['Ra']
Ra_opts = tuple(i * Ra_ref for i in (1.0, 0.5, 2.0))
Da_ref = SYSTEM_A_REFERENCE['Da']
Da_opts = tuple(i * Da_ref for i in (1.0, 0.1))# 0.1, 10.0))
sr_ref = SYSTEM_A_REFERENCE['sr']
sr_opts = tuple(i * sr_ref for i in (1, ))

SCRIPT = './scripts/run_system_a.py'
OVERWRITE = False

NX = 200
NY = 200
N_STOP = 99999
T_STOP = 120.0

WRITE_DELTA = (5, 3) if N_STOP > 100 else 1
NICE = 0
LOG = True

DIR_PARAMS = FrozenDict(
    dir_root='./data',
    dir_params=(*SYSTEM_A_REFERENCE.keys(), 'Nx', 'Ny'),
    dir_prefix=f'n_stop={N_STOP}',
    dir_suffix=None,
    dir_datetime=False,
    dir_uid=True,
)
CONFIG_PARAMS = FrozenDict(
    store_delta=None, 
    write_delta=WRITE_DELTA,
    **DIR_PARAMS,
)

SIMULATION_PARAMS = FrozenDict(
    **SYSTEM_A_REFERENCE,
    Nx=NX,
    Ny=NY,
    courant_adv=0.75,
    courant_diff=0.75,
    courant_reac=0.1,
    c_stabilization=None,
    c_limits=True,
)
RUN_PARAMS = dict(
    n_stop=N_STOP,
    t_stop=T_STOP,
    dt_init=1e-6,
    n_init=20,
)
DEL_XDMF = False

opt_names = ('Ra', 'Da', 'sr')
run_opts = []
for Ra, Da, sr in combine_options(Ra_opts, Da_opts, sr_opts, link=False):
    simulation_params = SIMULATION_PARAMS.replace(
        Ra=Ra,
        Da=Da,
        sr=sr,
    )
    dir_path_partial = create_dir_path(
        simulation_params,
        **DIR_PARAMS.replace(dir_uid=False)
    )
    if not OVERWRITE and not dir_path_exists(dir_path_partial, glob_suffix='*'):
        print(f'Scheduled to run {opt_names} = {Ra, Da, sr}')
        run_opts.append((Ra, Da, sr))
    else:
        print(f'Already run {opt_names} = {Ra, Da, sr}')


for Ra, Da, sr in run_opts:
    simulation_params = SIMULATION_PARAMS.replace(
        Ra=Ra,
        Da=Da,
        sr=sr,
    )
    cli_dict = {
        **CONFIG_PARAMS,
        **simulation_params,
        **RUN_PARAMS,
    }
    cli_kwargs = [(f'--{k}', repr(v)) for k, v in cli_dict.items()]
    p = subprocess.Popen(
        [
            "nice", "-n", repr(NICE),
            "nohup", "python", SCRIPT, 
            "--Ra", repr(Ra),
            "--Da", repr(Da),
            "--sr", repr(sr),
            "--delete_xdmf", repr(DEL_XDMF),
            *[i for pair in cli_kwargs for i in pair],
        ],
        start_new_session=True,
        stdin=subprocess.DEVNULL,
        stdout=subprocess.DEVNULL if not LOG else open(f"run_full.log", "a"),
        stderr=subprocess.DEVNULL,
        close_fds=True,
    )
    print(f"PID: {p.pid}  {opt_names} = {Ra, Da, sr}")

Scheduled to run ('Ra', 'Da', 'sr') = (1000.0, 100.0, 0.2)
Scheduled to run ('Ra', 'Da', 'sr') = (1000.0, 10.0, 0.2)
Scheduled to run ('Ra', 'Da', 'sr') = (500.0, 100.0, 0.2)
Scheduled to run ('Ra', 'Da', 'sr') = (500.0, 10.0, 0.2)
Scheduled to run ('Ra', 'Da', 'sr') = (2000.0, 100.0, 0.2)
Scheduled to run ('Ra', 'Da', 'sr') = (2000.0, 10.0, 0.2)
PID: 72409  ('Ra', 'Da', 'sr') = (1000.0, 100.0, 0.2)
PID: 72410  ('Ra', 'Da', 'sr') = (1000.0, 10.0, 0.2)
PID: 72411  ('Ra', 'Da', 'sr') = (500.0, 100.0, 0.2)
PID: 72412  ('Ra', 'Da', 'sr') = (500.0, 10.0, 0.2)
PID: 72413  ('Ra', 'Da', 'sr') = (2000.0, 100.0, 0.2)
PID: 72415  ('Ra', 'Da', 'sr') = (2000.0, 10.0, 0.2)
